# Grad-CAM for the HAM10000 ConvNeXt-Tiny model (Team 16)

Explains the saved **ConvNeXt-Tiny @ 224 px** run (`v5_convnext_tiny224_..._seed42`, kept epoch 20, test macro-F1 0.7783).
No retraining. It needs the files the training notebook already saved: `best_<tag>.pth` and `probs_<tag>.npz`.

**Run order:** mount Drive and set paths, write the module, load model and data, then run the figure cells.
The test set is only used for explanation here. Nothing is tuned on it.

In [1]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

# ---------------- paths (same as the training notebook) ----------------
DATA_DIR  = "/content/drive/MyDrive/HAM10000"
SPLIT_DIR = DATA_DIR
OUT_DIR   = "/content/drive/MyDrive/HAM10000_model/imbalance_runs"
GRADCAM_DIR = "/content/drive/MyDrive/HAM10000_model/gradcam"      # figures are saved here
TAG_PATTERN = "v5_convnext_tiny224*seed42"                          # picks best_<tag>.pth and probs_<tag>.npz

# image folders: the local copy is faster if it still exists in this session
import os
IMAGE_DIRS = ["/content/ham_images"] if os.path.isdir("/content/ham_images") else \
             [f"{DATA_DIR}/HAM10000_images_part_1", f"{DATA_DIR}/HAM10000_images_part_2"]
os.makedirs(GRADCAM_DIR, exist_ok=True)
print(IMAGE_DIRS)

GPU 0: Tesla T4 (UUID: GPU-4fd0db59-7148-ed30-fc50-f6ef1ccb3358)
Mounted at /content/drive
['/content/drive/MyDrive/HAM10000/HAM10000_images_part_1', '/content/drive/MyDrive/HAM10000/HAM10000_images_part_2']


In [2]:
%%writefile gradcam_convnext.py
"""
Grad-CAM and Grad-CAM++ for the HAM10000 v5 ConvNeXt-Tiny model (Team 16).

Matches the training notebook (p1_HAM10000_v5_backbones):
  * torchvision convnext_tiny, head replaced by Linear(768, 7)
  * checkpoint best_<tag>.pth: keys model_state_dict (EMA weights), class_to_idx, backbone, img_size
  * 224 px, ImageNet mean/std, class order = alphabetical (akiec, bcc, bkl, df, mel, nv, vasc)
  * the reported test numbers use flip TTA (4 flips, logits averaged), so by default the
    heatmap is also averaged over the 4 flips (each CAM is un-flipped before averaging).

Target layer: model.features[-1]  (stage 4, 7x7 map at 224 px).
For a finer but less class-specific map use model.features[5] (stage 3, 14x14).
"""
import os
import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import convnext_tiny

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
FULL_NAMES = {
    "akiec": "actinic keratosis / Bowen's", "bcc": "basal cell carcinoma", "bkl": "benign keratosis",
    "df": "dermatofibroma", "mel": "melanoma", "nv": "melanocytic nevus", "vasc": "vascular lesion",
}
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
IMG_SIZE = 224

_to_tensor = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
# flips are their own inverse, so the same function maps an image view or a CAM back
FLIPS = [lambda t: t, lambda t: t.flip(3), lambda t: t.flip(2), lambda t: t.flip((2, 3))]


# ----------------------------------------------------------------------------- model / data
def find_checkpoint(out_dir, pattern="best_v5_convnext_tiny224*seed42.pth"):
    hits = sorted(glob.glob(os.path.join(out_dir, pattern)))
    if not hits:
        raise FileNotFoundError(f"No checkpoint matching {pattern} in {out_dir}")
    return hits[-1]


def load_model(ckpt_path, device):
    """Rebuild convnext_tiny with a 7-class head and load the saved (EMA) weights."""
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)   # file holds a numpy array, so weights_only=False
    assert ck.get("backbone", "convnext_tiny") == "convnext_tiny", f"checkpoint backbone is {ck.get('backbone')}"
    assert ck.get("class_to_idx", CLASS_TO_IDX) == CLASS_TO_IDX, "class order differs from the notebook"
    m = convnext_tiny(weights=None)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, len(CLASSES))
    m.load_state_dict(ck["model_state_dict"])
    return m.to(device).eval(), ck


def find_image_paths(dirs):
    paths = {}
    for d in dirs:
        for p in glob.glob(os.path.join(d, "*.jpg")):
            paths[os.path.splitext(os.path.basename(p))[0]] = p
    return paths


def load_test_df(split_dir, image_dirs):
    df = pd.read_csv(os.path.join(split_dir, "test_split.csv"))     # CSV order = order of probs_*.npz
    paths = find_image_paths(image_dirs)
    df["image_path"] = df["image_id"].map(paths)
    df["label"] = df["dx"].map(CLASS_TO_IDX)
    assert df["image_path"].notna().all(), "some test images were not found; check image_dirs"
    return df


def load_image(path, size=IMG_SIZE):
    """Returns (PIL image at size x size, normalised tensor 3 x size x size)."""
    pil = Image.open(path).convert("RGB").resize((size, size), Image.BILINEAR)
    return pil, _to_tensor(pil)


@torch.no_grad()
def predict_probs(model, x, tta=True):
    views = FLIPS if tta else FLIPS[:1]
    logits = torch.stack([model(f(x)) for f in views]).mean(0)
    return logits.softmax(1).cpu().numpy()


# ----------------------------------------------------------------------------- Grad-CAM engine
class CamExplainer:
    """One forward/backward per view gives both Grad-CAM and Grad-CAM++."""

    def __init__(self, model, target_layer=None, tta=True):
        self.model, self.tta = model.eval(), tta
        self.layer = target_layer if target_layer is not None else model.features[-1]
        self.acts, self.grads = None, None
        self._h = self.layer.register_forward_hook(self._on_forward)

    def close(self):
        self._h.remove()

    def _on_forward(self, module, inp, out):
        self.acts = out
        if out.requires_grad:                                  # skipped during no_grad forwards
            out.register_hook(lambda g: setattr(self, "grads", g))

    @staticmethod
    def _gradcam(A, G):
        w = G.mean(dim=(2, 3), keepdim=True)                   # channel weights = mean gradient
        return F.relu((w * A).sum(1, keepdim=True))

    @staticmethod
    def _gradcampp(A, G):
        g2, g3 = G ** 2, G ** 3
        denom = 2 * g2 + A.sum(dim=(2, 3), keepdim=True) * g3
        denom = torch.where(denom != 0, denom, torch.ones_like(denom))
        w = ((g2 / denom) * F.relu(G)).sum(dim=(2, 3), keepdim=True)
        return F.relu((w * A).sum(1, keepdim=True))

    def explain(self, x, target=None):
        """
        x: normalised tensor (1, 3, H, W) on the model's device.
        target: class index to explain; None = the model's own (TTA) prediction.
        Returns dict: probs (7,), pred, target, gradcam (H, W), gradcampp (H, W), both scaled to 0..1.
        """
        H, W = x.shape[-2:]
        views = FLIPS if self.tta else FLIPS[:1]
        probs = predict_probs(self.model, x, self.tta)[0]
        pred = int(probs.argmax())
        tgt = pred if target is None else int(target)
        cams = {"gradcam": 0, "gradcampp": 0}
        for f in views:
            self.model.zero_grad(set_to_none=True)
            with torch.enable_grad():
                out = self.model(f(x))
                out[0, tgt].backward()                         # raw logit of the target class
            A, G = self.acts.detach(), self.grads.detach()
            for name, fn in (("gradcam", self._gradcam), ("gradcampp", self._gradcampp)):
                c = F.interpolate(fn(A, G), size=(H, W), mode="bilinear", align_corners=False)
                cams[name] = cams[name] + f(c)                 # f() undoes the flip on the CAM
        res = {"probs": probs, "pred": pred, "target": tgt}
        for name, c in cams.items():
            c = c[0, 0].cpu().numpy()
            res[name] = (c - c.min()) / (c.max() - c.min() + 1e-8)
        self.acts = self.grads = None
        return res


# ----------------------------------------------------------------------------- case selection (uses saved probs)
def top_confusions(probs, y, n=4):
    """The n most frequent (true, pred) mistakes, as [(true_idx, pred_idx, count)]."""
    pred = probs.argmax(1)
    cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
    for t, p in zip(y, pred):
        cm[t, p] += 1
    np.fill_diagonal(cm, 0)
    order = np.dstack(np.unravel_index(np.argsort(-cm, axis=None), cm.shape))[0][:n]
    return [(int(t), int(p), int(cm[t, p])) for t, p in order]


def _pick(idx, score, k, pick, seed):
    """pick: random | confident (highest prob first) | uncertain (lowest prob first) | mixed (half confident, half uncertain)."""
    if len(idx) == 0:
        return idx
    if pick == "random":
        return np.random.default_rng(seed).permutation(idx)[:k]
    order = idx[np.argsort(-score[idx])]
    if pick == "confident":
        return order[:k]
    if pick == "uncertain":
        return order[::-1][:k]
    if pick == "mixed":
        a = (k + 1) // 2
        return np.concatenate([order[:a], order[a:][::-1][:k - a]])
    raise ValueError("pick must be random, confident, uncertain or mixed")


def correct_cases(probs, y, cls, k=6, pick="random", seed=0):
    """Correctly classified test images of class cls. pick='random' avoids cherry-picking."""
    pred = probs.argmax(1)
    idx = np.flatnonzero((y == cls) & (pred == cls))
    return _pick(idx, probs[:, cls], k, pick, seed)


def confusion_cases(probs, y, true_cls, pred_cls, k=6, pick="random", seed=0):
    """Test images of class true_cls that the model labelled pred_cls."""
    pred = probs.argmax(1)
    idx = np.flatnonzero((y == true_cls) & (pred == pred_cls))
    return _pick(idx, probs[:, pred_cls], k, pick, seed)


# ----------------------------------------------------------------------------- running + stats
def center_fraction(cam, radius=0.4):
    """Share of heatmap mass inside a central disc (radius as a fraction of image size).
    A rough check for attention on image borders; it is NOT a lesion segmentation."""
    h, w = cam.shape
    yy, xx = np.mgrid[:h, :w]
    mask = (yy - h / 2) ** 2 + (xx - w / 2) ** 2 <= (radius * min(h, w)) ** 2
    return float(cam[mask].sum() / (cam.sum() + 1e-8))


def run_cases(explainer, test_df, idxs, device, target="pred"):
    """target='pred' explains the predicted class; target='true' explains the true class."""
    recs = []
    for i in idxs:
        r = test_df.iloc[int(i)]
        pil, x = load_image(r["image_path"])
        tgt = None if target == "pred" else int(r["label"])
        out = explainer.explain(x.unsqueeze(0).to(device), target=tgt)
        recs.append(dict(idx=int(i), image_id=r["image_id"], true=int(r["label"]), rgb=np.asarray(pil), **out))
    return recs


def overlay(rgb, cam, alpha=0.45):
    heat = (matplotlib.colormaps["jet"](cam)[..., :3] * 255).astype(np.float32)
    return ((1 - alpha) * rgb.astype(np.float32) + alpha * heat).clip(0, 255).astype(np.uint8)


def _caption(r):
    ok = "OK" if r["true"] == r["pred"] else "WRONG"
    return (f"{r['image_id']}\ntrue {CLASSES[r['true']]} | pred {CLASSES[r['pred']]} "
            f"({r['probs'][r['pred']]:.2f}) {ok}")


# ----------------------------------------------------------------------------- figures
def plot_triplet(rec, method="gradcam", save_path=None):
    """Original | heatmap | overlay for one image."""
    cam = rec[method]
    fig, ax = plt.subplots(1, 3, figsize=(10, 3.6))
    ax[0].imshow(rec["rgb"]); ax[0].set_title("input")
    ax[1].imshow(cam, cmap="jet", vmin=0, vmax=1); ax[1].set_title(f"{method} for '{CLASSES[rec['target']]}'")
    ax[2].imshow(overlay(rec["rgb"], cam)); ax[2].set_title("overlay")
    for a in ax:
        a.axis("off")
    fig.suptitle(_caption(rec), fontsize=9)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_grid(recs, method="gradcam", title="", save_path=None):
    """Top row: inputs. Bottom row: overlays."""
    n = len(recs)
    if n == 0:
        print(f"{title}: no cases"); return
    fig, ax = plt.subplots(2, n, figsize=(2.5 * n, 5.6), squeeze=False)
    for k, r in enumerate(recs):
        ax[0, k].imshow(r["rgb"]); ax[0, k].set_title(_caption(r), fontsize=7)
        ax[1, k].imshow(overlay(r["rgb"], r[method]))
        ax[1, k].set_title(f"{method} -> {CLASSES[r['target']]}", fontsize=7)
        ax[0, k].axis("off"); ax[1, k].axis("off")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_compare(recs, title="Grad-CAM vs Grad-CAM++", save_path=None):
    """Rows: input, Grad-CAM, Grad-CAM++."""
    n = len(recs)
    fig, ax = plt.subplots(3, n, figsize=(2.5 * n, 7.8), squeeze=False)
    for k, r in enumerate(recs):
        ax[0, k].imshow(r["rgb"]); ax[0, k].set_title(_caption(r), fontsize=7)
        ax[1, k].imshow(overlay(r["rgb"], r["gradcam"]))
        ax[2, k].imshow(overlay(r["rgb"], r["gradcampp"]))
        for j in range(3):
            ax[j, k].axis("off")
    ax[1, 0].text(-0.05, 0.5, "Grad-CAM", transform=ax[1, 0].transAxes, rotation=90, va="center", ha="right")
    ax[2, 0].text(-0.05, 0.5, "Grad-CAM++", transform=ax[2, 0].transAxes, rotation=90, va="center", ha="right")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


# ----------------------------------------------------------------------------- sanity check + bulk export
def check_against_saved(model, test_df, saved_probs, device, n=64, tta=True):
    """Re-predict the first n test images and compare with probs_<tag>.npz. Expect ~100% agreement and a
    small probability difference (training evaluated in fp16 and from a pre-resized cache)."""
    got = []
    for i in range(n):
        _, x = load_image(test_df.iloc[i]["image_path"])
        got.append(predict_probs(model, x.unsqueeze(0).to(device), tta)[0])
    got = np.stack(got)
    agree = float((got.argmax(1) == saved_probs[:n].argmax(1)).mean())
    diff = float(np.abs(got - saved_probs[:n]).max())
    print(f"checkpoint check on {n} test images: prediction agreement {agree:.1%}, max |prob diff| {diff:.3f}")
    return agree, diff


def export_all(explainer, test_df, device, out_dir, method="gradcam", every=1):
    """Saves input|overlay PNG for every `every`-th test image and a CSV of predictions and heatmap stats."""
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    for i in range(0, len(test_df), every):
        r = run_cases(explainer, test_df, [i], device)[0]
        side = np.concatenate([r["rgb"], overlay(r["rgb"], r[method])], axis=1)
        name = f"{r['image_id']}_true-{CLASSES[r['true']]}_pred-{CLASSES[r['pred']]}.png"
        Image.fromarray(side).save(os.path.join(out_dir, name))
        rows.append(dict(image_id=r["image_id"], true=CLASSES[r["true"]], pred=CLASSES[r["pred"]],
                         p_pred=float(r["probs"][r["pred"]]), p_true=float(r["probs"][r["true"]]),
                         center_fraction_gradcam=center_fraction(r["gradcam"]),
                         center_fraction_gradcampp=center_fraction(r["gradcampp"])))
        if len(rows) % 100 == 0:
            print("exported", len(rows), flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out_dir, "gradcam_index.csv"), index=False)
    return df


# ----------------------------------------------------------------------------- error gallery (feature 2)
def _idx(c):
    return CLASS_TO_IDX[c] if isinstance(c, str) else int(c)


def error_table(test_df, probs, y):
    """Every misclassified test image with the model's confidence in its prediction and in the true class."""
    pred = probs.argmax(1)
    w = np.flatnonzero(pred != y)
    return pd.DataFrame({
        "image_id": test_df["image_id"].to_numpy()[w],
        "true": [CLASSES[i] for i in y[w]], "pred": [CLASSES[i] for i in pred[w]],
        "p_pred": probs[w, pred[w]].round(4), "p_true": probs[w, y[w]].round(4)})


def plot_confusion(probs, y, highlight=(), save_path=None):
    """Confusion matrix (counts; colour = share of the true class). Pairs in `highlight` get a red box."""
    pred = probs.argmax(1)
    n = len(CLASSES)
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(y, pred):
        cm[t, p] += 1
    fig, ax = plt.subplots(figsize=(6.2, 5.4))
    ax.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                    color="white" if cm[i, j] / max(cm[i].sum(), 1) > 0.5 else "black")
    for t, p in highlight:
        ax.add_patch(plt.Rectangle((_idx(p) - 0.5, _idx(t) - 0.5), 1, 1, fill=False, ec="red", lw=2.5))
    ax.set_xticks(range(n)); ax.set_xticklabels(CLASSES, rotation=45)
    ax.set_yticks(range(n)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"Test confusion matrix (red = pairs in the gallery)\naccuracy {np.mean(pred == y):.4f}")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_error_gallery(test_df, probs, y, true_cls, pred_cls, k=8, pick="random", seed=0, ncols=4,
                       explainer=None, device=None, method="gradcam", conf_warn=0.8, save_path=None):
    """Real test images of class true_cls that the model labelled pred_cls, with the model's confidence.
    If explainer is given, each image is shown with its Grad-CAM overlay instead of the plain photo."""
    t, p = _idx(true_cls), _idx(pred_cls)
    pred = probs.argmax(1)
    n_pair = int(((y == t) & (pred == p)).sum())
    n_true = int((y == t).sum())
    idxs = confusion_cases(probs, y, t, p, k=k, pick=pick, seed=seed)
    title = (f"{CLASSES[t]} predicted as {CLASSES[p]}: {n_pair} of {n_true} {CLASSES[t]} test images "
             f"({100 * n_pair / max(n_true, 1):.1f}%) | showing {len(idxs)}, pick={pick}")
    if len(idxs) == 0:
        print(title, "-> no such errors in the test set"); return []
    if explainer is not None:
        recs = run_cases(explainer, test_df, idxs, device)
        imgs = [overlay(r["rgb"], r[method]) for r in recs]
    else:
        imgs = [np.asarray(load_image(test_df.iloc[int(i)]["image_path"])[0]) for i in idxs]
    nrows = int(np.ceil(len(idxs) / ncols))
    fig, ax = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.5 * nrows), squeeze=False)
    for a in ax.ravel():
        a.axis("off")
    for k_, i in enumerate(idxs):
        a = ax[k_ // ncols, k_ % ncols]
        a.imshow(imgs[k_])
        pp, pt = probs[i, p], probs[i, t]
        a.set_title(f"{test_df.iloc[int(i)]['image_id']}\nmodel: {CLASSES[p]} {pp:.0%} | true {CLASSES[t]}: {pt:.0%}",
                    fontsize=8, color="darkred" if pp >= conf_warn else "black")
    fig.suptitle(title + f"\n(red title = model was at least {conf_warn:.0%} sure of the wrong label)", fontsize=10)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return idxs


Writing gradcam_convnext.py


In [3]:
import glob, numpy as np, torch
import gradcam_convnext as gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path  = sorted(glob.glob(f"{OUT_DIR}/*best_{TAG_PATTERN}.pth"))[-1]   # "Copy of " prefix is fine
probs_path = sorted(glob.glob(f"{OUT_DIR}/*probs_{TAG_PATTERN}*.npz*"))[-1]  # also finds a .npz.zip (np.load reads it directly)
print("checkpoint:", os.path.basename(ckpt_path)); print("probs     :", os.path.basename(probs_path))

model, ck = gc.load_model(ckpt_path, device)
print("kept epoch:", ck["best_epoch"], "| img_size:", ck["img_size"])

test_df = gc.load_test_df(SPLIT_DIR, IMAGE_DIRS)
z = np.load(probs_path)
probs, y = z["test_probs"], z["test_y"]
assert len(test_df) == len(probs) and (test_df["label"].to_numpy() == y).all(), "test CSV order does not match saved probs"

from sklearn.metrics import accuracy_score, f1_score
pred = probs.argmax(1)
print(f"saved test accuracy {accuracy_score(y, pred):.4f} | macro-F1 {f1_score(y, pred, average='macro'):.4f}   (notebook reported 0.8346 / 0.7783)")
gc.check_against_saved(model, test_df, probs, device, n=64)     # expect ~100% agreement

checkpoint: best_v5_convnext_tiny224_rs_inverse_auto_cap1200_floor500_strong_ls0.1_sch-ema-tta_ep25_best_seed42.pth
probs     : probs_v5_convnext_tiny224_rs_inverse_auto_cap1200_floor500_strong_ls0.1_sch-ema-tta_ep25_best_seed42 (2).npz.zip
kept epoch: 20 | img_size: 224
saved test accuracy 0.8346 | macro-F1 0.7783   (notebook reported 0.8346 / 0.7783)
checkpoint check on 64 test images: prediction agreement 93.8%, max |prob diff| 0.173


(0.9375, 0.172654390335083)

## 1 · One image, step by step
Explains the model's **predicted** class. Change `IDX`, or set `target='true'` to see what supports the true class.

In [4]:
explainer = gc.CamExplainer(model, tta=True)        # layer = model.features[-1] (7x7). For a finer map: target_layer=model.features[5]

# default: first nv image that the model called mel (the biggest error in the reports)
nv, mel = gc.CLASS_TO_IDX["nv"], gc.CLASS_TO_IDX["mel"]
IDX = int(gc.confusion_cases(probs, y, nv, mel, k=1, pick="confident")[0])
rec = gc.run_cases(explainer, test_df, [IDX], device, target="pred")[0]
gc.plot_triplet(rec, "gradcam", save_path=f"{GRADCAM_DIR}/single_{rec['image_id']}.png")

## 2 · Per-class view: correctly classified images
`PICK='random'` is the honest choice for the report (no cherry-picking). Use `'confident'` only for illustration.

In [5]:
PICK, K, METHOD = "random", 6, "gradcam"          # METHOD: "gradcam" or "gradcampp"
for c, name in enumerate(gc.CLASSES):
    idxs = gc.correct_cases(probs, y, c, k=K, pick=PICK, seed=0)
    recs = gc.run_cases(explainer, test_df, idxs, device)
    gc.plot_grid(recs, METHOD, f"{name} ({gc.FULL_NAMES[name]}): correctly classified",
                 save_path=f"{GRADCAM_DIR}/correct_{name}.png")

## 3 · Errors: where does the model look when it is wrong?
Takes the most frequent mistakes from the saved test predictions (for example nv predicted as mel).

In [6]:
for t, p, n in gc.top_confusions(probs, y, n=4):
    idxs = gc.confusion_cases(probs, y, t, p, k=6, pick=PICK, seed=0)
    recs = gc.run_cases(explainer, test_df, idxs, device)                 # heatmap for the (wrong) predicted class
    gc.plot_grid(recs, METHOD, f"true {gc.CLASSES[t]} -> predicted {gc.CLASSES[p]}  ({n} test images)",
                 save_path=f"{GRADCAM_DIR}/error_{gc.CLASSES[t]}_as_{gc.CLASSES[p]}.png")

## 4 · Grad-CAM vs Grad-CAM++
Same images, both methods. Grad-CAM++ usually spreads over more of the lesion when the evidence is in several places.

In [7]:
mix = np.concatenate([gc.correct_cases(probs, y, mel, 3, "random", 1), gc.confusion_cases(probs, y, nv, mel, 3, "random", 1),
                      gc.correct_cases(probs, y, gc.CLASS_TO_IDX["bcc"], 2, "random", 1)])
gc.plot_compare(gc.run_cases(explainer, test_df, mix, device), save_path=f"{GRADCAM_DIR}/compare_methods.png")

## 5 · Optional: export every test image
Saves an input|overlay PNG for each image and `gradcam_index.csv` (prediction, probabilities, and a rough *center_fraction*: how much of the heatmap sits in the central disc; low values suggest attention on the border). Takes several minutes on a T4. Set `EVERY = 5` for a quick subset.

In [8]:
EXPORT_ALL, EVERY = False, 1
if EXPORT_ALL:
    idx_df = gc.export_all(explainer, test_df, device, f"{GRADCAM_DIR}/all_test", method="gradcam", every=EVERY)
    print(idx_df.groupby(["true", "pred"]).center_fraction_gradcam.mean().round(2).head(20))

## How to read the maps (put these caveats in the report)
- Red = region that pushed the score of the explained class up. It is **evidence used by the model, not a proof of correctness** and not a lesion segmentation.
- The last ConvNeXt stage is 7x7 at 224 px, so maps are blurry by design. Use `target_layer=model.features[5]` for a finer 14x14 map.
- Maps are averaged over the same 4 flips as the reported test predictions, so the explained class equals the class in your confusion matrix.
- Check for shortcuts: attention on skin-marker ink, rulers, hair, vignette corners or image borders. `center_fraction` helps flag these.
- `df` (10 test images) and `vasc` (16) are tiny, so show several examples before drawing a conclusion.

# Feature 2 · Error gallery
Real test images of the main mistakes, each with the model's confidence. It uses the saved test predictions (`probs_*.npz`), so the counts match your confusion matrix exactly.
Run the setup cells above first (paths, module, load). Red title = the model was at least 80% sure of the wrong label.

In [9]:
# ---- settings ----
EXTRA_PAIRS = [("nv", "mel"), ("akiec", "bkl")]   # always shown (true -> predicted); edit freely
N_AUTO      = 4          # plus this many of the most frequent errors found automatically
K           = 8          # images per pair
GALLERY_PICK = "random"  # "random" | "confident" | "uncertain" | "mixed"
SHOW_GRADCAM = False     # True = show the Grad-CAM overlay instead of the plain photo
GALLERY_DIR = f"{GRADCAM_DIR}/error_gallery"
os.makedirs(GALLERY_DIR, exist_ok=True)

pairs = list(EXTRA_PAIRS)
for t, p, n in gc.top_confusions(probs, y, n=N_AUTO):
    if (gc.CLASSES[t], gc.CLASSES[p]) not in pairs:
        pairs.append((gc.CLASSES[t], gc.CLASSES[p]))
print("pairs:", pairs)

pairs: [('nv', 'mel'), ('akiec', 'bkl'), ('nv', 'bkl'), ('mel', 'nv'), ('bkl', 'mel')]


In [10]:
# confusion matrix with the gallery pairs boxed in red
gc.plot_confusion(probs, y, highlight=pairs, save_path=f"{GALLERY_DIR}/confusion_matrix.png")

In [11]:
# one gallery per pair
ex = gc.CamExplainer(model, tta=True) if SHOW_GRADCAM else None
for t, p in pairs:
    gc.plot_error_gallery(test_df, probs, y, t, p, k=K, pick=GALLERY_PICK, seed=0, explainer=ex, device=device,
                          save_path=f"{GALLERY_DIR}/errors_{t}_as_{p}.png")

In [12]:
# every misclassified test image with its confidences (for the report appendix)
errs = gc.error_table(test_df, probs, y)
errs.to_csv(f"{GALLERY_DIR}/all_errors.csv", index=False)
print(len(errs), "errors of", len(test_df), "test images")
print("\nconfident mistakes (model >= 80% sure of the wrong label):", int((errs.p_pred >= 0.8).sum()))
print(errs.groupby(["true", "pred"]).size().sort_values(ascending=False).head(10))

168 errors of 1016 test images

confident mistakes (model >= 80% sure of the wrong label): 55
true   pred
nv     mel     52
       bkl     21
mel    nv      20
bkl    nv       8
       mel      8
nv     bcc      8
akiec  bkl      6
       mel      6
bkl    bcc      6
mel    bkl      6
dtype: int64
